In [0]:
SELECT current_schema();

In [0]:
USE suresh_db.dinedash;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/orders/orders_2024_01.json',format => 'json')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/orders/orders_2024_02.json',format => 'json')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_customers.csv',format => 'csv')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_delivery_agents.csv',format => 'csv')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_locations.csv',format => 'csv')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_menu_items.csv',format => 'csv')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_restaurants.csv',format => 'csv')
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_tmp_view AS
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_customers.csv',format => 'csv');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW delivery_agents_tmp_view AS
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_delivery_agents.csv',format => 'csv');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW locations_tmp_view AS
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_locations.csv',format => 'csv');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW menu_items_tmp_view AS
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_menu_items.csv',format => 'csv');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW restaurants_tmp_view AS
SELECT *
FROM read_files('/Volumes/suresh_db/dinedash/source/dimensions/dim_restaurants.csv',format => 'csv');

In [0]:
%sql
DESCRIBE customers_tmp_view

In [0]:
SET TIME ZONE '+05:30';
SELECT current_timestamp();

In [0]:
%sql
CREATE OR REPLACE TABLE customers_bronze(
    customer_id	STRING,
    name STRING NOT NULL,
    email STRING,
    dob	DATE,
    signup_date	DATE,
    location_id STRING,
    ingestion_time TIMESTAMP,
    source_file_name STRING,
    _rescued_data STRING
)
USING DELTA;

In [0]:
INSERT INTO customers_bronze
SELECT
    customer_id,
    name,
    email,
    dob,
    signup_date,
    location_id,
    current_timestamp(),
    input_file_name(),
    _rescued_data
FROM customers_tmp_view;

In [0]:
SELECT *
FROM customers_bronze
LIMIT 10;

In [0]:
DESCRIBE restaurants_tmp_view;

In [0]:
CREATE OR REPLACE TABLE restaurants_bronze(
    restaurant_id STRING,
    restaurant_name STRING,
    cuisines STRING,
    location_id	STRING,
    rating DOUBLE,
    delivery_fee DOUBLE,
    ingestion_time TIMESTAMP,
    source_file_name STRING,
    _rescued_data STRING
)
USING DELTA;

In [0]:
INSERT INTO TABLE restaurants_bronze
SELECT
    restaurant_id,
    name,
    cuisines,
    location_id,
    rating,
    delivery_fee,
    current_timestamp(),
    input_file_name(),
    _rescued_data
FROM restaurants_tmp_view;

In [0]:
SELECT *
FROM restaurants_bronze
LIMIT 10;

In [0]:
DESCRIBE locations_tmp_view;

In [0]:
CREATE OR REPLACE TABLE locations_bronze(
    location_id STRING,
    area STRING,
    city STRING,
    state STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    ingestion_time TIMESTAMP,
    source_file_name STRING,
    _rescued_data STRING
)
USING DELTA;

In [0]:
INSERT into locations_bronze
SELECT
    location_id,
    area,
    city,
    state,
    latitude,
    longitude,
    current_timestamp(),
    input_file_name(),
    _rescued_data
FROM locations_tmp_view;

In [0]:
SELECT *
FROM locations_bronze
LIMIT 10;

In [0]:
DESCRIBE delivery_agents_tmp_view;

In [0]:
CREATE OR REPLACE TABLE delivery_agents_bronze(
    agent_id STRING,
    name STRING,
    phone STRING,
    rating DOUBLE,
    ingestion_time TIMESTAMP,
    source_file_name STRING,
    _rescued_data STRING
)
USING DELTA;

In [0]:
INSERT INTO delivery_agents_bronze
SELECT
    agent_id,
    name,
    phone,
    rating,
    current_timestamp(),
    input_file_name(),
    _rescued_data
FROM delivery_agents_tmp_view;

In [0]:
SELECT *
FROM delivery_agents_bronze
LIMIT 10;

In [0]:
DESCRIBE menu_items_tmp_view;

In [0]:
CREATE OR REPLACE TABLE menu_items_bronze(
    item_id STRING,
    restaurant_id STRING,
    item_name STRING,
    price DOUBLE,
    category STRING,
    ingestion_time TIMESTAMP,
    source_file_name STRING,
    _rescued_data STRING
)
USING DELTA;

In [0]:
INSERT INTO TABLE menu_items_bronze
SELECT
    item_id,
    restaurant_id,
    item_name,
    price,
    category,
    current_timestamp(),
    input_file_name(),
    _rescued_data
FROM menu_items_tmp_view;

In [0]:
SELECT *
FROM menu_items_bronze
LIMIT 10;

In [0]:
DROP TABLE IF EXISTS orders_bronze;
CREATE TABLE orders_bronze;

In [0]:
COPY INTO orders_bronze
FROM (
    SELECT
        *,
        current_timestamp() as ingestion_time,
        _metadata.file_path as source_file_name
    FROM '/Volumes/suresh_db/dinedash/source/orders/orders_2024_01.json'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema'='true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
SELECT 
    source_file_name,
    COUNT(*) as records
FROM orders_bronze
GROUP BY source_file_name;

In [0]:
COPY INTO orders_bronze
FROM (
    SELECT
        *,
        current_timestamp() as ingestion_time,
        _metadata.file_path as source_file_name
    FROM '/Volumes/suresh_db/dinedash/source/orders/orders_2024_02.json'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema'='true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
SELECT 
    source_file_name,
    COUNT(*) as records
FROM orders_bronze
GROUP BY source_file_name;

In [0]:
COPY INTO orders_bronze
FROM (
    SELECT
        *,
        current_timestamp() as ingestion_time,
        _metadata.file_path as source_file_name
    FROM '/Volumes/suresh_db/dinedash/source/orders/orders_2024_03.json'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema'='true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
SELECT 
    source_file_name,
    COUNT(*) as records
FROM orders_bronze
GROUP BY source_file_name;